# Assignment 05: Training Loop from Scratch (100 points)

**Unit 06: Programming PyTorch | AI 310**

In this assignment, you will implement a complete training pipeline from scratch — the full forward/backward/update cycle. No shortcuts, no `model.fit()`. This is exactly the skill tested in USAAIO Round 2.

**Notation**:
- $\theta$ = model parameters
- $\eta$ = learning rate
- $\mathcal{L}$ = loss function

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries.

---

## Part 1 (15 points, coding)

Implement a `train_one_epoch` function that performs one full pass over the training data.

Requirements:
- Set model to training mode
- For each batch: forward pass, compute loss, zero gradients, backward, step
- Return average loss and accuracy for the epoch

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def train_one_epoch(model, loader, criterion, optimizer):
    """
    Train for one epoch.
    
    Args:
        model: nn.Module
        loader: DataLoader yielding (features, labels)
        criterion: loss function
        optimizer: torch.optim optimizer
    
    Returns:
        avg_loss: float, average loss over all samples
        accuracy: float, fraction of correct predictions
    """
    pass

In [ ]:
""" END OF THIS PART """
# Quick test
torch.manual_seed(42)
test_model = nn.Linear(10, 3)
test_ds = torch.utils.data.TensorDataset(
    torch.randn(100, 10), torch.randint(0, 3, (100,))
)
test_loader = DataLoader(test_ds, batch_size=32)
test_opt = torch.optim.SGD(test_model.parameters(), lr=0.01)
test_crit = nn.CrossEntropyLoss()

loss, acc = train_one_epoch(test_model, test_loader, test_crit, test_opt)
assert isinstance(loss, float) and loss > 0
assert isinstance(acc, float) and 0 <= acc <= 1
print(f"Part 1 passed! Loss: {loss:.4f}, Acc: {acc:.4f}")

---

## Part 2 (15 points, coding)

Implement an `evaluate` function for validation/testing.

Requirements:
- Set model to evaluation mode
- Use `torch.no_grad()` context
- Return average loss and accuracy
- Do NOT update model parameters

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def evaluate(model, loader, criterion):
    """
    Evaluate model on a dataset.
    
    Args:
        model: nn.Module
        loader: DataLoader
        criterion: loss function
    
    Returns:
        avg_loss: float
        accuracy: float
    """
    pass

In [ ]:
""" END OF THIS PART """
# Save model state before evaluation
state_before = {k: v.clone() for k, v in test_model.state_dict().items()}
val_loss, val_acc = evaluate(test_model, test_loader, test_crit)
state_after = test_model.state_dict()

# Parameters should NOT change during evaluation
for key in state_before:
    assert torch.allclose(state_before[key], state_after[key]), \
        f"Parameter {key} changed during evaluation!"

assert isinstance(val_loss, float) and val_loss > 0
assert isinstance(val_acc, float) and 0 <= val_acc <= 1
print(f"Part 2 passed! Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

---

## Part 3 (20 points, coding)

Implement a `train` function with **early stopping** and **checkpointing**.

Requirements:
- Train for up to `max_epochs` epochs
- After each epoch, evaluate on validation set
- If validation loss improves, save the model state dict (in memory, not to disk)
- If validation loss does not improve for `patience` consecutive epochs, stop training
- After training, restore the best model state
- Return the training history (lists of train/val loss and accuracy per epoch)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def train(model, train_loader, val_loader, max_epochs, lr, patience=5):
    """
    Full training pipeline with early stopping.
    
    Args:
        model: nn.Module
        train_loader, val_loader: DataLoaders
        max_epochs: maximum number of epochs
        lr: learning rate
        patience: epochs to wait before early stopping
    
    Returns:
        model: nn.Module with best weights loaded
        history: dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
                 each mapping to a list of per-epoch values
    """
    pass

In [ ]:
""" END OF THIS PART """
torch.manual_seed(42)
# Create a simple classification problem
X = torch.randn(500, 20)
y = (X[:, 0] + X[:, 1] > 0).long()  # simple linear boundary
full_ds = torch.utils.data.TensorDataset(X, y)
train_ds, val_ds = random_split(full_ds, [400, 100])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=100)

model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 2))
model, history = train(model, train_loader, val_loader, max_epochs=100, lr=1e-2, patience=10)

assert 'train_loss' in history and 'val_loss' in history
assert 'train_acc' in history and 'val_acc' in history
assert len(history['train_loss']) <= 100, "Should stop before 100 epochs (early stopping)"
assert len(history['train_loss']) > 5, "Should train for at least a few epochs"
# Validation accuracy should be reasonable for this simple problem
assert history['val_acc'][-1] > 0.6, f"Val acc too low: {history['val_acc'][-1]}"
print(f"Part 3 passed! Trained for {len(history['train_loss'])} epochs")
print(f"Final val acc: {history['val_acc'][-1]:.4f}")

---

## Part 4 (25 points, coding)

**Regression training loop.**

Train a neural network to fit the function $f(x) = \sin(x) \cdot e^{-0.1x}$ on $x \in [0, 10]$.

Requirements:
1. Create training data: 500 points uniformly in $[0, 10]$ with Gaussian noise ($\sigma = 0.05$)
2. Model: 2-hidden-layer MLP with 64 units and ReLU activation
3. Loss: MSELoss
4. Optimizer: Adam with lr=1e-3
5. Train for 500 epochs
6. Store final MSE loss on a **noiseless** test set (200 points) as `test_mse`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# 1. Create training data
# 2. Build model
# 3. Training loop (500 epochs)
# 4. Evaluate on noiseless test set

test_mse = ...  # float

In [ ]:
""" END OF THIS PART """
assert isinstance(test_mse, float)
assert test_mse < 0.01, f"Test MSE too high: {test_mse:.6f}. Model not fitting well enough."
print(f"Part 4 passed! Test MSE: {test_mse:.6f}")

---

## Part 5 (25 points, coding)

**Manual SGD implementation.**

Implement the training loop using **manual parameter updates** instead of `optimizer.step()`. This tests your understanding of what the optimizer does under the hood.

Requirements:
1. Create a linear model: $\hat{y} = Wx + b$
2. Generate data from $y = 3x_1 - 2x_2 + 1 + \epsilon$ where $\epsilon \sim \mathcal{N}(0, 0.1)$
3. Implement SGD manually: $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$
4. Do NOT use any `torch.optim` optimizer
5. Train for 1000 steps with lr=0.01
6. Store learned weight as `learned_w` (shape `(1, 2)`) and bias as `learned_b` (shape `(1,)`)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# True: y = 3*x1 - 2*x2 + 1

# YOUR CODE HERE
# - Create data
# - Initialize W and b as tensors with requires_grad=True
# - Training loop with MANUAL updates (no optimizer)

learned_w = ...  # shape (1, 2), should be close to [3, -2]
learned_b = ...  # shape (1,), should be close to [1]

In [ ]:
""" END OF THIS PART """
assert learned_w.shape == (1, 2) or learned_w.shape == torch.Size([1, 2])
assert learned_b.shape == (1,) or learned_b.shape == torch.Size([1])

w_vals = learned_w.detach().squeeze()
b_val = learned_b.detach().squeeze()

print(f"Learned W: [{w_vals[0]:.3f}, {w_vals[1]:.3f}] (target: [3, -2])")
print(f"Learned b: {b_val:.3f} (target: 1.0)")

assert abs(w_vals[0].item() - 3.0) < 0.3, f"w1 should be ~3, got {w_vals[0].item()}"
assert abs(w_vals[1].item() - (-2.0)) < 0.3, f"w2 should be ~-2, got {w_vals[1].item()}"
assert abs(b_val.item() - 1.0) < 0.3, f"b should be ~1, got {b_val.item()}"
print("Part 5 passed!")